# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities using their `@id` fields for precise data mapping.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. It may require a restart after installation in some environments.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available Record Sets with their @id fields
record_sets = dataset.record_sets
print("Available Record Sets:")
record_set_ids = []
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '<no name>')} | description: {rs.get('description', '')}")
    record_set_ids.append(rs['@id'])

# Inspect fields for each record set
for rs in record_sets:
    print(f"\nFields for Record Set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for f in fields:
            # Each field is a dict or reference
            if isinstance(f, dict):
                print(f"  - Field @id: {f.get('@id',str(f))} | name: {f.get('name','<no name>')} | dataType: {f.get('dataType','')} | description: {f.get('description','')}")
            else:
                print(f"  - Field @id: {str(f)}")
    else:
        print("  <No fields defined>")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis. Each record set and field is referenced by its `@id`.

In [ ]:
# If record_sets are available, extract them by their @id
# Let's use the first available record set for demonstration
dataframes = {}
if record_set_ids:
    # For demonstration, take all available record set @id
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nColumns in Record Set @id {rs_id}:")
        print(df.columns.tolist())
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric values, normalization, and grouping.

In [ ]:
# Select a record set and field for numeric analysis (ensure that the columns exist and are numeric)
import numpy as np

if dataframes:
    # Use the first loaded record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"\nAnalyzing Record Set: {record_set_id}")

    # Attempt to infer a numeric field for demonstration
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Try to group by a non-numeric field if available
        non_numeric_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
# Visualization: Numeric field distribution
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes:
    df = dataframes[record_set_id]
    if numeric_fields:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field_id} in Record Set '@id': {record_set_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric fields to plot.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset using the `mlcroissant` library, referencing all entities via their `@id`. 

- We examined record sets and their fields by `@id`.
- Loaded data into Pandas DataFrames.
- Performed data extraction, transformation, and basic EDA, including normalization and group-wise summaries.
- Visualized numeric field distributions.

This structure enables reproducible and schema-driven FAIR dataset analysis across Croissant-compliant datasets.